# Task 3 — Gender Audience-Auxiliary E10 Experiment

This Colab runner trains only Gender E10. It keeps the exact E6 GeM model and all official five-way Gender rows, then adds one training-only three-way catalogue-audience head. It never retrains E1–E9.

## 1. Safe Colab setup

Run on a GPU runtime. The repository update stops on local changes. Training data is extracted into Colab-local storage; evidence and the registry stay in Drive.

In [ ]:
from pathlib import Path
import math
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=cwd, check=True)

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This runner must run in Google Colab.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError(
            "The Colab clone has local changes. Save them first; no switch or merge was attempted."
        )
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

print(f"Repository ready: {REPO_DIR} ({BRANCH})")

In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}
image_dirs = (
    teacher_dir / "train/images_train",
    teacher_dir / "test/images_test",
)

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [
        name for name in names if Path(name).is_absolute() or ".." in Path(name).parts
    ]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/")
        and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(
        path.suffix.lower() in image_suffixes
        for image_dir in image_dirs
        for path in image_dir.glob("*")
    )
    needs_extract = current_images != expected_images or not all(
        path.is_file() for path in required_files
    )
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(
    path.suffix.lower() in image_suffixes
    for image_dir in image_dirs
    for path in image_dir.glob("*")
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found "
        f"{actual_images:,}; missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
DRIVE_TASK_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print("Output folders are ready.")

## 2. Exact E6 parents and zero-step audit

The frozen IDs must match the completed E6 folds in Drive. The audit checks the saved error counts, the five-to-three label mapping, and helper-label coverage before any optimizer exists.

In [ ]:
from fashion.train.task3_e10 import write_task3_e10_prerun_evidence
from fashion.train.task3_experiments import (
    GENDER_E10_AUXILIARY_LOSS_WEIGHT,
    GENDER_E10_PEAK_MEMORY_LIMIT_BYTES,
    GENDER_E10_PRIMARY_LOSS_WEIGHT,
    audit_completed_registry_rows,
    check_task3_child_setup,
    latest_completed_gender_e6_parent_run_ids,
    run_task3_child_cv,
)

GENDER_E6_PARENT_RUN_IDS = (
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f0_s2753_a8c09286451b_20260831T090059Z0bab1f",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f1_s2753_a8c09286451b_20260831T090940Z6e10b5",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f2_s2753_a8c09286451b_20260831T091823Zabb677",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f3_s2753_a8c09286451b_20260831T092710Zee7c6a",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f4_s2753_a8c09286451b_20260831T093553Z63b5fd",
)

resolved_gender = latest_completed_gender_e6_parent_run_ids(output_root=DRIVE_TASK_DIR)
if resolved_gender != GENDER_E6_PARENT_RUN_IDS:
    raise RuntimeError("Resolved Gender E6 parents differ from the frozen tuple.")

gender_e10_check = check_task3_child_setup(
    "gender_audience_aux",
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    root=REPO_DIR,
    device_name="cuda",
)
if gender_e10_check["optimizer_steps"] != 0:
    raise RuntimeError("Preflight unexpectedly reported an optimizer step.")
if gender_e10_check["parameter_count"] != 390_952:
    raise RuntimeError("E10 parameter count changed.")
if gender_e10_check["auxiliary_output_shape"] != [2, 3]:
    raise RuntimeError("E10 audience-head shape changed.")
if gender_e10_check["training_selection_strategy"] != "all":
    raise RuntimeError("E10 must keep every official fold-training row.")
if gender_e10_check["zero_step_peak_memory_bytes"] >= GENDER_E10_PEAK_MEMORY_LIMIT_BYTES:
    raise RuntimeError("E10 failed the frozen zero-step 2 GiB memory check.")
if not math.isclose(
    gender_e10_check["primary_loss_weight"], GENDER_E10_PRIMARY_LOSS_WEIGHT
) or not math.isclose(
    gender_e10_check["auxiliary_loss_weight"], GENDER_E10_AUXILIARY_LOSS_WEIGHT
):
    raise RuntimeError("E10 loss weights changed.")
print("E10 model contract passed zero-step preflight.")
print("Zero-step peak memory bytes:", gender_e10_check["zero_step_peak_memory_bytes"])

In [ ]:
GENDER_E6_AGGREGATE = (
    DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender/aggregate"
)
GENDER_E6_OOF = GENDER_E6_AGGREGATE / "oof_predictions.csv"
GENDER_E6_METRICS = GENDER_E6_AGGREGATE / "metrics.json"
missing_audit_inputs = [
    str(path) for path in (GENDER_E6_OOF, GENDER_E6_METRICS) if not path.is_file()
]
if missing_audit_inputs:
    raise FileNotFoundError(f"Missing saved E6 evidence: {missing_audit_inputs}")

e10_prerun = write_task3_e10_prerun_evidence(
    splits_path=REPO_DIR / "data/processed/splits.csv",
    parent_prediction_path=GENDER_E6_OOF,
    parent_metrics_path=GENDER_E6_METRICS,
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    output_dir=DRIVE_TASK_DIR / "e10_prerun",
)
if e10_prerun["optimizer_steps"] != 0:
    raise RuntimeError("The E10 audit unexpectedly reported an optimizer step.")
if not e10_prerun["verified"] or not all(e10_prerun["checks"].values()):
    raise RuntimeError("The E10 parent evidence did not verify.")
print("Clean E6 errors:", e10_prerun["clean_errors"])
print("Audience/Unisex E6 errors:", e10_prerun["audience_or_unisex_errors"])
print("Focus share:", e10_prerun["audience_or_unisex_share_of_clean_errors"])
print("E10 audit saved to Drive; optimizer steps: 0")

## 3. Train Gender E10

This is the only training call. It runs all five official folds from scratch. The three-way head helps training, but saved validation predictions and submission output remain the required five Gender classes.

In [ ]:
gender_e10 = run_task3_child_cv(
    "gender_audience_aux",
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    output_root=DRIVE_TASK_DIR,
    folds=range(5),
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    device_name="cuda",
)
gender_registry_audit = audit_completed_registry_rows(
    DRIVE_REGISTRY, gender_e10["fold_run_ids"]
)
print("Gender E10 complete:", gender_e10["metrics_path"])
print(gender_registry_audit)

## 4. Stop and return to the main notebook

Do not run another seed or add another change here. Return the saved E10 aggregate to Notebook 04 and apply the frozen five-way, audience-error, fold, and robustness gates. The separate Usage label audit remains deferred.